# 05 — SQL/PostgreSQL — GUIA VIVO — COLAB

**MATERIAL DIDÁTICO FICTÍCIO/SINTÉTICO, NÃO OFICIAL; versão de revisão.** Fonte de temas: transcrições e apresentações SC TEC T8 Semanas 08–11, com divergências documentadas. Os dados são inventados; não são bases da professora ou do mini-projeto da Semana 7.

**Como executar:** ambiente Google Colab (Python online; runtime temporário). Execute células Python de cima para baixo. As células SQL usam **SQLite em memória como simulação conceitual**, não PostgreSQL real. As células Mongo usam **objetos Python**, não mongosh/Compass. Use os scripts `labs/postgresql/` e `labs/mongodb/` somente em servidores de treino autorizados.

Acesse separadamente `docs/ambientacao/fontes-oficiais-bancos.md`, `primeiro-acesso-postgresql.md` e `primeiro-acesso-mongodb.md` para downloads, cadastro de conexão e reconhecimento da interface. **SQL é a linguagem de PostgreSQL; mongosh usa JavaScript, diferente de Java; Python/Java são integrações por drivers, não pré-condição de uso da GUI.**

CHECKPOINT: COMPREENDER (explicar), EXECUTAR (prever/rodar em ambiente correto), DIAGNOSTICAR (corrigir interpretação).

## 🔁 Você já viu isso

Este eixo reaproveita listas/dicionários, operações, Pandas, tipos, filtros e qualidade. Não trate coincidência de nome como equivalência de esquema.

In [ ]:
import sqlite3
cn=sqlite3.connect(":memory:")
cn.row_factory=sqlite3.Row
cn.executescript("""
PRAGMA foreign_keys=ON;
CREATE TABLE clientes(id INTEGER PRIMARY KEY,nome TEXT,estado TEXT);
CREATE TABLE produtos(id TEXT PRIMARY KEY,nome TEXT,preco NUMERIC);
CREATE TABLE pedidos(id INTEGER PRIMARY KEY,cliente_id INTEGER REFERENCES clientes(id),dia TEXT,status TEXT);
CREATE TABLE itens(pedido_id INTEGER REFERENCES pedidos(id),produto_id TEXT REFERENCES produtos(id),qtd INTEGER,preco NUMERIC,desconto NUMERIC,PRIMARY KEY(pedido_id,produto_id));
INSERT INTO clientes VALUES(1,'Lia','SP'),(2,'Raul','RJ'),(3,'Bia','MG');
INSERT INTO produtos VALUES('P01','Caderno',10),('P02','Caneta',20);
INSERT INTO pedidos VALUES(101,1,'2026-11-05','concluido'),(102,2,'2026-11-09','concluido');
INSERT INTO itens VALUES(101,'P01',2,10,0),(101,'P02',1,20,0),(102,'P01',3,5,10);
""")
print("Simulação SQLite na memória — NÃO é PostgreSQL conectado.")
assert cn.execute("SELECT COUNT(*) FROM clientes").fetchone()[0]==3

## ✅ CHECKPOINT 1 — Servidor, cliente e linguagens

[AULA M1S08] PostgreSQL é o servidor; pgAdmin é o cliente gráfico; Query Tool envia SQL. O editor VS Code e o Colab executam Python em seus próprios ambientes, não criam automaticamente um servidor PostgreSQL. Primeiro contato: identificar host, porta, banco, usuário e conexão autorizada. O laboratório real deve começar em pgAdmin com `SELECT current_database();` e `SELECT current_user;`. Porta usual 5432 não é garantia de configuração.

In [ ]:
print('SQLite local:',cn.execute('SELECT 1').fetchone()[0]);assert cn.execute('SELECT 1').fetchone()[0]==1

### Domínio CP1

COMPREENDER a distinção servidor/cliente. EXECUTAR o exemplo SQLite e, futuramente, a consulta PostgreSQL no banco autorizado. DIAGNOSTICAR erro de conexão sem atribuí-lo ao SQL.

**Dica de Ouro:** antes da execução, identifique dados de entrada, unidade de análise, saída esperada e limite do método.

## ✅ CHECKPOINT 2 — Tabelas, chaves e consultas de leitura

[AULA M1S08–10] Tabela tem linhas/colunas. PK identifica registro e FK preserva referência. `SELECT nome FROM clientes WHERE estado='SP' ORDER BY nome` significa escolher coluna, origem, filtro e ordenação. Não confunda índice de Pandas com PK de banco. No servidor real, usar schema e banco corretos.

In [ ]:
rows=cn.execute("SELECT nome FROM clientes WHERE estado='SP'").fetchall();print([r['nome'] for r in rows]);assert rows[0]['nome']=='Lia'

### Domínio CP2

COMPREENDER PK e FK. EXECUTAR SELECT com filtro. DIAGNOSTICAR tabela ausente em banco/esquema errado.

**Dica de Ouro:** antes da execução, identifique dados de entrada, unidade de análise, saída esperada e limite do método.

## ✅ CHECKPOINT 3 — JOIN e granularidade

[AULA M1S10–11] Em JOIN pedido→itens cada item resulta uma linha. INNER exige par; LEFT preserva linhas da esquerda. Cliente Bia não possui pedido. `COUNT(*)` após JOIN dá 3 linhas; `COUNT(DISTINCT pedido_id)` dá 2 pedidos. WHERE sobre coluna de uma tabela opcional pode eliminar as linhas preservadas pelo LEFT. RIGHT/FULL são mostrados na fonte SQL PostgreSQL, mas não homologados aqui em SQLite.

In [ ]:
r=cn.execute('SELECT COUNT(*) a, COUNT(DISTINCT p.id) b FROM pedidos p JOIN itens i ON i.pedido_id=p.id').fetchone();assert (r['a'],r['b'])==(3,2);print(dict(r))

### Domínio CP3

COMPREENDER 1:N. EXECUTAR JOIN com previsão. DIAGNOSTICAR duplicação de indicadores.

**Dica de Ouro:** antes da execução, identifique dados de entrada, unidade de análise, saída esperada e limite do método.

## ✅ CHECKPOINT 4 — GROUP BY, HAVING e métricas

[AULA M1S11] WHERE filtra linhas; GROUP BY organiza grupos; HAVING filtra os grupos. Pedidos, unidades e valor de itens são medidas distintas. `qtd * preco * (1 - desconto/100)` produz valor didático, não lucro ou pagamento recebido. Valor esperado: pedido 101 R$40; pedido 102 R$13,50; total R$53,50 e 6 unidades.

In [ ]:
r=cn.execute('SELECT COUNT(DISTINCT pedido_id) n,SUM(qtd) u,SUM(qtd*preco*(1-desconto/100.0)) v FROM itens').fetchone();assert (r['n'],r['u'],round(r['v'],2))==(2,6,53.5);print(dict(r))

### Domínio CP4

COMPREENDER unidade de análise e filtro de status. EXECUTAR soma por pedido. DIAGNOSTICAR receita inflada por JOIN.

**Dica de Ouro:** antes da execução, identifique dados de entrada, unidade de análise, saída esperada e limite do método.

## ✅ CHECKPOINT 5 — Subconsultas e condições de negócio

[AULA M1S11] Subconsulta pode devolver valor escalar, conjunto IN ou tabela derivada com alias. Acima da média global NÃO prova outlier quando categorias diferem. Cliente com UM pedido acima de um limite é diferente de cliente cuja SOMA supera o limite. Primeiro escreva a pergunta de negócio em linguagem natural.

In [ ]:
q='SELECT pedido_id FROM (SELECT pedido_id,SUM(qtd*preco*(1-desconto/100.0)) total FROM itens GROUP BY pedido_id) r WHERE total>20';rs=cn.execute(q).fetchall();assert [x[0] for x in rs]==[101];print('Pedidos >20:',[x[0] for x in rs])

### Domínio CP5

COMPREENDER limiar por pedido vs total cliente. EXECUTAR tabela derivada. DIAGNOSTICAR média fora de contexto.

**Dica de Ouro:** antes da execução, identifique dados de entrada, unidade de análise, saída esperada e limite do método.

## ✅ CHECKPOINT 6 — CTE e legibilidade

[AULA M1S11] WITH nome AS (...) nomeia uma consulta intermediária e facilita decompor um relatório; não garante aceleração por si. A CTE deve preservar a granularidade (um resumo por pedido). Um JOIN posterior pode voltar a multiplicar linhas se for 1:N.

In [ ]:
q='WITH r AS (SELECT pedido_id,SUM(qtd) unidades FROM itens GROUP BY pedido_id) SELECT COUNT(*) FROM r';assert cn.execute(q).fetchone()[0]==2;print('CTE: dois resumos de pedido')

### Domínio CP6

COMPREENDER a finalidade do WITH. EXECUTAR resumo e previsão. DIAGNOSTICAR perda de granularidade.

**Dica de Ouro:** antes da execução, identifique dados de entrada, unidade de análise, saída esperada e limite do método.

## ✅ CHECKPOINT 7 — Datas e interpretação temporal

[AULA M1S11] PostgreSQL possui `DATE_TRUNC('month', data_pedido)::date`, usado para agrupamento por mês; não é comando SQLite. O exemplo SQLite usa substring textual SOMENTE para ilustrar o mês. Comparar novembro parcial com outubro completo sem ajustar janelas induz conclusão errada. CONCAT e SUBSTRING organizam apresentação, não validam cadastros.

In [ ]:
r=cn.execute('SELECT substr(dia,1,7) mes,COUNT(*) n FROM pedidos GROUP BY substr(dia,1,7)').fetchall();print([tuple(x) for x in r]);assert len(r)==1

### Domínio CP7

COMPREENDER mês parcial. EXECUTAR previsão de agrupamento. DIAGNOSTICAR diferenças de dialeto SQL.

**Dica de Ouro:** antes da execução, identifique dados de entrada, unidade de análise, saída esperada e limite do método.

## ✅ CHECKPOINT 8 — Transações, integridade e comunicação

[AULA M1S08–11; COMPLEMENTO] Antes de escrever dados: confirmar conexão, isolamento, PK/FK, status, quantidade de linhas afetadas e reversibilidade. PostgreSQL suporta transações BEGIN/ROLLBACK em muitos casos; isto não torna qualquer ação inofensiva. Nem o notebook nem a presença do SQL provam sua execução em servidor real. Não usar scripts da professora em base de produção.

In [ ]:
assert cn.execute('PRAGMA foreign_key_check').fetchall()==[];print('Sem violações de FK nos dados fictícios SQLite')

### Domínio CP8

COMPREENDER integridade e escopo. EXECUTAR checagens. DIAGNOSTICAR consulta correta com interpretação comercial incorreta.

**Dica de Ouro:** antes da execução, identifique dados de entrada, unidade de análise, saída esperada e limite do método.

## Revisão acumulativa

Explique, sem consultar: servidor vs cliente, sintaxe da linguagem vs formato de dados, documentos vs linhas, aula observada vs slide planejado, exemplo sintético vs tarefa formal. A execução Python deste Guia **não valida** os scripts em PostgreSQL ou MongoDB.

**Próximo passo:** abra o Caderno Prático espelhado. As soluções aparecem após espaço de tentativa; só consulte depois de responder.